# Notebook 4 — Threshold Optimisation
Automatically searches τ ∈ [0.40, 0.95] using Precision · Recall · F1 · ROC-AUC.

No hardcoded threshold — optimal value stored to `models/optimal_threshold.json`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

from modules.anomaly_detector import load_isolation_forest, batch_scores
from modules.confidence import escalation_score, normalise_error
from modules.preprocessing import FEATURE_COLS
from config import (
    DATA_PATH, THRESHOLD_PATH,
    THRESHOLD_SEARCH_LOW, THRESHOLD_SEARCH_HIGH, THRESHOLD_STEPS
)

print('Ready.')

## 1. Build Escalation Scores for Full Dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
numeric_feat_cols = [c for c in FEATURE_COLS if c in df.columns]
X = df[numeric_feat_cols].values
y_true = df['anomaly_label'].values

clf = load_isolation_forest()
a_scores = batch_scores(clf, X)   # normalised anomaly scores

# Simulate prediction errors (use rolling diff as proxy before LSTM is loaded)
traffic = df['traffic_mbps'].values
pred_errors = np.abs(np.diff(traffic, prepend=traffic[0])) / (traffic.max() + 1e-8)
pred_errors = np.clip(pred_errors, 0, 1)

# QoS risk proxy: packet_loss column
qos_risk = df['packet_loss'].values if 'packet_loss' in df.columns else np.zeros(len(df))
qos_risk = np.clip(qos_risk, 0, 1)

esc_scores = np.array([
    escalation_score(pred_errors[i], a_scores[i], qos_risk[i])
    for i in range(len(df))
])

print(f'Escalation score stats — min:{esc_scores.min():.3f}  '
      f'mean:{esc_scores.mean():.3f}  max:{esc_scores.max():.3f}')

## 2. Grid Search over τ

In [ ]:
thresholds = np.linspace(THRESHOLD_SEARCH_LOW, THRESHOLD_SEARCH_HIGH, THRESHOLD_STEPS)
records = []

for tau in thresholds:
    y_pred = (esc_scores > tau).astype(int)
    prec   = precision_score(y_true, y_pred, zero_division=0)
    rec    = recall_score(y_true, y_pred, zero_division=0)
    f1     = f1_score(y_true, y_pred, zero_division=0)
    try:
        auc = roc_auc_score(y_true, esc_scores)
    except Exception:
        auc = 0.0
    records.append({'tau': tau, 'precision': prec, 'recall': rec, 'f1': f1, 'auc': auc})

res_df = pd.DataFrame(records)
best_row = res_df.loc[res_df['f1'].idxmax()]
optimal_tau = float(best_row['tau'])

print(f'Optimal τ = {optimal_tau:.4f}  '
      f'(F1={best_row["f1"]:.4f}, P={best_row["precision"]:.4f}, R={best_row["recall"]:.4f})')

## 3. Save Optimal Threshold

In [ ]:
THRESHOLD_PATH.parent.mkdir(parents=True, exist_ok=True)
payload = {
    'optimal_threshold': optimal_tau,
    'f1':        float(best_row['f1']),
    'precision': float(best_row['precision']),
    'recall':    float(best_row['recall']),
    'auc':       float(best_row['auc']),
}
with open(THRESHOLD_PATH, 'w') as f:
    json.dump(payload, f, indent=2)
print(f'Saved to {THRESHOLD_PATH}')
print(json.dumps(payload, indent=2))

## 4. Visualisation

In [ ]:
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=('Precision', 'Recall', 'F1 Score', 'ROC-AUC'))
metrics_plot = [('precision','#00B4D8'), ('recall','#F77F00'),
                ('f1','#2DC653'), ('auc','#9B5DE5')]
positions = [(1,1),(1,2),(2,1),(2,2)]

for (r,c), (metric, color) in zip(positions, metrics_plot):
    fig.add_trace(go.Scatter(x=res_df['tau'], y=res_df[metric],
                             line=dict(color=color), name=metric), row=r, col=c)
    fig.add_vline(x=optimal_tau, line_dash='dash', line_color='white', row=r, col=c)

fig.update_layout(template='plotly_dark', height=500,
                  title=f'Threshold Optimisation — Optimal τ = {optimal_tau:.4f}')
fig.show()
print('Notebook 4 complete.')